<a href="https://colab.research.google.com/github/kodurumeghana3/Data-Science-Internship---February-2026/blob/main/NLPbTask2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Understanding

In [5]:
import pandas as pd
import kagglehub
import os

# Download and Load the Dataset
print("Downloading dataset via Kagglehub...")
folder_path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Dataset downloaded to folder:", folder_path)
csv_file_path=os.path.join(folder_path, "IMDB Dataset.csv")
print("\nLoading dataset...")
df=pd.read_csv(csv_file_path)

# Explore Number of Samples
print("\n----- Dataset Size -----")
print("Total number of rows and columns:", df.shape)
print("Total number of reviews:", len(df))

# Class Distribution
print("\n----- Class Distribution -----")
sentiment_counts=df['sentiment'].value_counts()
print(sentiment_counts)

# Sample Texts
print("\n----- Sample Positive Review -----")
positive_sample=df[df['sentiment']=='positive']['review'].iloc[0]
print(positive_sample)

print("\n----- Sample Negative Review -----")
negative_sample=df[df['sentiment']=='negative']['review'].iloc[0]
print(negative_sample)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Dataset downloaded to folder: /kaggle/input/imdb-dataset-of-50k-movie-reviews

Loading dataset...

----- Dataset Size -----
Total number of rows and columns: (50000, 2)
Total number of reviews: 50000

----- Class Distribution -----
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

----- Sample Positive Review -----
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Peni

NLP Precprocessing

In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


print("Downloading NLTK tools...")
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer=WordNetLemmatizer()
stop_words=set(stopwords.words('english'))

# Function
def clean_text(text):
    # Lowercasing
    text=text.lower()

    # Handling special characters/URLs & HTML
    text=re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text=re.sub(r'<.*?>', ' ', text)

    # Removing punctuation
    text=re.sub(r'[^a-z\s]', '', text)

    # Tokenization
    words=word_tokenize(text)

    # Removing Stopwords & Lemmatization
    cleaned_words=[]
    for word in words:
        if word not in stop_words:
            root_word=lemmatizer.lemmatize(word)
            cleaned_words.append(root_word)
    return " ".join(cleaned_words)

# Test the Function
sample_review="I LOVED this movie!!!  The acting was running beautifully, link: http://imdb.com."
print("\n----- Testing the Cleaning Function -----")
print("Original Text:", sample_review)
print("Cleaned Text :", clean_text(sample_review))

# Apply to the Entire Dataset
print("\nCleaning the entire dataset")
df['cleaned_review']=df['review'].apply(clean_text)

print("\n----- First 5 rows of Cleaned Data -----")
print(df[['review', 'cleaned_review']].head())


----- Testing the Cleaning Function -----
Original Text: I LOVED this movie!!!  The acting was running beautifully, link: http://imdb.com.
Cleaned Text : loved movie acting running beautifully link

Cleaning the entire dataset


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



----- First 5 rows of Cleaned Data -----
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                      cleaned_review  
0  one reviewer mentioned watching oz episode you...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically there family little boy jake think t...  
4  petter matteis love time money visually stunni...  


Feature Engineering

In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

corpus=df['cleaned_review'].tolist()
# Bag of Words (BoW)
print("\nCreating Bag of Words matrix...")
bow_vectorizer = CountVectorizer(max_features=5000)
X_bow=bow_vectorizer.fit_transform(corpus)
print("Bag of Words Matrix Shape:", X_bow.shape)
# Result should be (50000, 5000) -> 50,000 reviews, 5,000 word counts

# TF-IDF
tfidf_vectorizer=TfidfVectorizer(max_features=5000)
X_tfidf=tfidf_vectorizer.fit_transform(corpus)
print("TF-IDF Matrix Shape:", X_tfidf.shape)

# Words the vectorizers decided were most important
print("\nSample of learned vocabulary (first 20 words):")
vocabulary=bow_vectorizer.get_feature_names_out()
print(vocabulary[:20])

print("\nSample of TF-IDF vocabulary:")
tfidf_vocab=tfidf_vectorizer.get_feature_names_out()
print(tfidf_vocab[:20])

print("\nConverting sentiment labels to numbers (1 = positive, 0 = negative)...")
df['sentiment_numeric']=df['sentiment'].map({'positive': 1, 'negative': 0})
y=df['sentiment_numeric'].values


Creating Bag of Words matrix...
Bag of Words Matrix Shape: (50000, 5000)
TF-IDF Matrix Shape: (50000, 5000)

Sample of learned vocabulary (first 20 words):
['aaron' 'abandoned' 'abc' 'ability' 'able' 'abrupt' 'absence' 'absent'
 'absolute' 'absolutely' 'absurd' 'absurdity' 'abuse' 'abused' 'abusive'
 'abysmal' 'academy' 'accent' 'accept' 'acceptable']

Sample of TF-IDF vocabulary:
['aaron' 'abandoned' 'abc' 'ability' 'able' 'abrupt' 'absence' 'absent'
 'absolute' 'absolutely' 'absurd' 'absurdity' 'abuse' 'abused' 'abusive'
 'abysmal' 'academy' 'accent' 'accept' 'acceptable']

Converting sentiment labels to numbers (1 = positive, 0 = negative)...


Model Building

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
print("Splitting data into Training and Testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} reviews")
print(f"Testing set size: {X_test.shape[0]} reviews")
print("\nInitializing models...")
log_model=LogisticRegression(max_iter=1000)
nb_model = MultinomialNB()
tree_model = DecisionTreeClassifier(random_state=42)
print("\nTraining models...")
print("\nTraining Logistic Regression...")
log_model.fit(X_train, y_train)

print("\nTraining Naive Bayes...")
nb_model.fit(X_train, y_train)

print("\nTraining Decision Tree...")
tree_model.fit(X_train, y_train)

print("\nAll models have been successfully trained!")

Splitting data into Training and Testing sets...
Training set size: 40000 reviews
Testing set size: 10000 reviews

Initializing models...

Training models...

Training Logistic Regression...

Training Naive Bayes...

Training Decision Tree...

All models have been successfully trained!


Model Evaluation

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

print("Starting Model Evaluation...\n")
log_predictions=log_model.predict(X_test)
nb_predictions=nb_model.predict(X_test)
tree_predictions=tree_model.predict(X_test)

def evaluate_model(y_true, y_pred, model_name):
    accuracy=accuracy_score(y_true, y_pred)
    precision=precision_score(y_true, y_pred)
    recall=recall_score(y_true, y_pred)
    f1=f1_score(y_true, y_pred)
    return {
        'Model': model_name,
        'Accuracy': round(accuracy, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1 Score': round(f1, 4)
    }

# Calculate Scores for Each Model
log_results=evaluate_model(y_test, log_predictions, 'Logistic Regression')
nb_results=evaluate_model(y_test, nb_predictions, 'Naive Bayes')
tree_results=evaluate_model(y_test, tree_predictions, 'Decision Tree')

# Compare Performance
all_results=[log_results, nb_results, tree_results]
comparison_df=pd.DataFrame(all_results)

print("\n----- Final Model Comparison -----")
print(comparison_df.to_string(index=False))

Starting Model Evaluation...


----- Final Model Comparison -----
              Model  Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.8860     0.8772  0.8998    0.8883
        Naive Bayes    0.8532     0.8529  0.8563    0.8546
      Decision Tree    0.7174     0.7229  0.7122    0.7175
